# Neural SDE Tutorial

**Phase 7.7: Neural Stochastic Differential Equations**

This tutorial introduces **Neural SDEs**: data-driven dynamics where drift \(\mu_\theta(S_t,t)\) and diffusion \(\sigma_\theta(S_t,t)\) are neural networks. You will:

1. **Build** a Neural SDE (drift/diffusion networks + solver)
2. **Simulate** paths with untrained (random) weights
3. **Train** the model on synthetic or historical paths
4. **Generate** paths with the trained model for scenarios or pricing

**References:** `docs/reference/models/neural_sde.md`, `docs/guides/models/training_neural_sde.md`

---

## 1. Setup and imports

In [1]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 4)
print("Setup complete.")

Setup complete.


## 2. What is a Neural SDE?

The underlying is modelled as:

$$dS_t = \mu_\theta(S_t, t)\,dt + \sigma_\theta(S_t, t)\,dW_t$$

- **Drift** \(\mu_\theta\) and **diffusion** \(\sigma_\theta\) are neural networks (e.g. MLPs).
- We **train** them on historical paths (or option data) so the model matches observed dynamics.
- **Use cases:** scenario generation, stress testing, data augmentation, pricing under learned dynamics.

## 3. Build the Neural SDE

We construct drift and diffusion networks, an SDE solver (Euler–Maruyama), and combine them into `NeuralSDEDynamics`.

In [2]:
from src.models.neural_sde import (
    NeuralSDEDynamics,
    NeuralDriftNetwork,
    NeuralDiffusionNetwork,
    EulerMaruyamaSolver,
)

drift_net = NeuralDriftNetwork(hidden_dims=[64, 64])
diffusion_net = NeuralDiffusionNetwork(hidden_dims=[64, 64])
solver = EulerMaruyamaSolver()

sde = NeuralSDEDynamics(
    drift_network=drift_net,
    diffusion_network=diffusion_net,
    solver=solver,
)

print("Neural SDE built:", sde)

TypeError: NeuralSDEDynamics.__init__() got an unexpected keyword argument 'solver'

## 4. Simulate paths (untrained)

With random initial weights, we can already simulate paths. They will not match any particular market until we train.

In [ ]:
paths_untrained = sde.simulate(
    S0=100.0,
    T=1.0,
    n_steps=252,
    n_paths=2000,
)

# paths shape: (n_paths, n_steps+1)
print("Paths shape:", paths_untrained.shape)

plt.figure(figsize=(10, 4))
for i in range(min(50, paths_untrained.shape[0])):
    plt.plot(paths_untrained[i], alpha=0.4, color='blue')
plt.xlabel("Time step")
plt.ylabel("Spot")
plt.title("Untrained Neural SDE paths (sample)")
plt.tight_layout()
plt.show()

## 5. Train on synthetic data

We generate **synthetic GBM paths** as "historical" data, then fit the Neural SDE so its simulated paths match (moment matching / pathwise loss).

In [ ]:
# Synthetic GBM paths as training data
n_train = 3000
n_steps = 50
S0, T = 100.0, 1.0
mu, sigma = 0.05, 0.20
dt = T / n_steps
rng = np.random.default_rng(42)
z = rng.standard_normal((n_train, n_steps))
log_returns = (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * z
historical_paths = S0 * np.exp(np.cumsum(np.concatenate([np.zeros((n_train, 1)), log_returns], axis=1), axis=1))

print("Historical paths shape:", historical_paths.shape)

In [ ]:
from src.models.neural_sde.training import NeuralSDETrainer, TrainingConfig, TrainingResult

config = TrainingConfig(
    n_epochs=50,
    learning_rate=1e-3,
    batch_size=32,
    n_sim_paths=500,
    n_sim_steps=50,
    patience=10,
    verbose=True,
)
trainer = NeuralSDETrainer(config=config)

result = trainer.fit(sde, historical_paths)
print("Final loss:", result.final_loss)
print("Converged:", result.converged)
if hasattr(result, 'summary'):
    print("Summary:", result.summary())

## 6. Generate paths after training

Use `PathGenerator` to produce paths from the trained model for scenario analysis or Monte Carlo pricing.

In [ ]:
from src.models.neural_sde.generation.generator import PathGenerator

generator = PathGenerator(sde, seed=123)
paths_trained = generator.generate(
    S0=100.0,
    T=1.0,
    n_steps=252,
    n_paths=2000,
)

plt.figure(figsize=(10, 4))
for i in range(min(50, paths_trained.shape[0])):
    plt.plot(paths_trained[i], alpha=0.4, color='green')
plt.xlabel("Time step")
plt.ylabel("Spot")
plt.title("Trained Neural SDE paths (sample)")
plt.tight_layout()
plt.show()

## 7. Run via orchestrator pipeline (optional)

For config-driven runs with logging and artifacts, use the `ml.train_neural_sde` pipeline. See `examples/pipelines/run_train_neural_sde.py` and `docs/guides/models/training_neural_sde.md`.

In [ ]:
# Uncomment to run the pipeline:
# from pathlib import Path
# from src.orchestrator.pipelines.ml.train_neural_sde import build_pipeline
# from src.orchestrator.core.pipeline import PipelineRunner
# from src.orchestrator.core.context import Context
# from src.orchestrator.config.schemas import RunConfig
# from src.orchestrator.artifacts.store import ArtifactStore
#
# config = RunConfig(
#     pipeline="ml.train_neural_sde",
#     params={"ml": {"neural_sde": {"n_paths": 2000, "n_steps": 50, "S0": 100.0, "n_epochs": 50}}},
# )
# ctx = Context(run_id="tutorial", cfg=config, artifact_store=ArtifactStore(artifacts_root=Path("artifacts")))
# ctx = PipelineRunner().run(build_pipeline(), ctx)
# print(ctx.state.get("neural_sde_training_result"))
print("Pipeline usage: see examples/pipelines/run_train_neural_sde.py")

## 8. Summary and further reading

- **Neural SDEs** learn drift and diffusion from data; use them for scenario generation, stress testing, and data augmentation.
- **Training:** `NeuralSDETrainer` with `TrainingConfig`; fit on historical paths (shape `(n_samples, n_steps+1)`).
- **Generation:** `PathGenerator(sde).generate(S0, T, n_steps, n_paths)`.
- **Pipeline:** `ml.train_neural_sde` for orchestrated runs.

**References:**
- [Neural SDE reference](../../reference/models/neural_sde.md)
- [Training Neural SDE guide](../../guides/models/training_neural_sde.md)
- Kidger et al. (2021) "Neural SDEs"; Gierjatowicz et al. (2020) "Robust pricing and hedging via neural SDEs"